# 自编码器降维：神经网络版的 PCA
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/降维 | 源文件：自编码器_降维.py | 核心功能：PyTorch 自编码器、编码器/解码器架构、与 PCA 对比

## 概述

自编码器（Autoencoder）用神经网络实现降维：编码器把高维数据压缩到低维"瓶颈层"，解码器从瓶颈层重建原始数据。训练目标是最小化重建误差。

与 PCA 的关键区别：自编码器的编码和解码函数是非线性的（通过 ReLU 等激活函数），因此可以捕捉 PCA 无法处理的非线性流形结构。事实上，**线性自编码器在数学上等价于 PCA**。

脚本用 PyTorch 实现了一个简单的自编码器，在 Iris 数据集上对比了自编码器和 PCA 的降维效果。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris, load_digits
# --- 导入库 ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

## 数学原理

### 1. 自编码器的结构

**代码对应**：自编码器由编码器和解码器组成，中间层为降维后的表示。

**编码器**：$\mathbf{z} = f_{\text{enc}}(\mathbf{x}) = \sigma(\mathbf{W}_e\mathbf{x} + \mathbf{b}_e)$

**解码器**：$\hat{\mathbf{x}} = f_{\text{dec}}(\mathbf{z}) = \sigma(\mathbf{W}_d\mathbf{z} + \mathbf{b}_d)$

其中 $\mathbf{z} \in \mathbb{R}^d$（$d < p$）为瓶颈层（降维表示），$\sigma$ 为激活函数。

### 2. 重构损失

**代码对应**：训练目标为最小化输入与重构之间的差异。

**MSE 损失**（连续数据）：

$$L = \frac{1}{n}\sum_{i=1}^{n}\|\mathbf{x}_i - \hat{\mathbf{x}}_i\|_2^2 = \frac{1}{n}\sum_{i=1}^{n}\|\mathbf{x}_i - f_{\text{dec}}(f_{\text{enc}}(\mathbf{x}_i))\|_2^2$$

**交叉熵损失**（二值数据）：

$$L = -\frac{1}{n}\sum_{i=1}^{n}\sum_{j=1}^{p}\left[x_{ij}\log\hat{x}_{ij} + (1-x_{ij})\log(1-\hat{x}_{ij})\right]$$

### 3. 自编码器与 PCA 的关系

线性自编码器（无激活函数、单隐藏层）等价于 PCA：

设 $\mathbf{W}_e \in \mathbb{R}^{d \times p}$，$\mathbf{W}_d \in \mathbb{R}^{p \times d}$，最小化 $\|\mathbf{X} - \mathbf{X}\mathbf{W}_e^T\mathbf{W}_d^T\|_F^2$ 的最优解为 $\mathbf{W}_e$ 的行空间等于 PCA 的前 $d$ 个主成分方向。

非线性自编码器（使用 ReLU/Sigmoid 激活）可以捕捉**非线性**降维结构，这是 PCA 做不到的。

### 4. 正则化变体

**去噪自编码器**（Denoising Autoencoder）：对输入添加噪声 $\tilde{\mathbf{x}} = \mathbf{x} + \epsilon$，训练目标为从噪声中恢复原始输入：

$$L = \|\mathbf{x} - f_{\text{dec}}(f_{\text{enc}}(\tilde{\mathbf{x}}))\|^2$$

这迫使编码器学习更鲁棒的特征。

**稀疏自编码器**：在损失中加入稀疏性惩罚：

$$L = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 + \lambda\sum_{j}\|z_j\|_1$$

### 5. 瓶颈维度的选择

瓶颈层维度 $d$ 越小，压缩比越大，信息损失越多。选择准则：
- $d$ 太大：无压缩效果，可能过拟合
- $d$ 太小：信息损失严重，重构质量差
- 通常通过重构误差的"拐点"选择，类似 PCA 的累积方差准则

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_tensor = torch.FloatTensor(X_train)
dataset = TensorDataset(X_tensor, X_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

### 2. 定义自编码器

运行 2. 定义自编码器 的代码，观察算法在该环节的行为。

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        # 编码器：输入 → 隐层 → 潜在表示
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        # 解码器：潜在表示 → 隐层 → 重建输入
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

    def encode(self, x):
        return self.encoder(x)

### 3. 训练自编码器

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
input_dim = X_train.shape[1]
hidden_dim = 8
latent_dim = 2  # 目标降维维度

model = Autoencoder(input_dim, hidden_dim, latent_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("=== 训练自编码器 ===")
n_epochs = 200
for epoch in range(n_epochs):
    epoch_loss = 0
    for batch_X, _ in loader:
# --- 继续 ---
        recon, _ = model(batch_X)
        loss = criterion(recon, batch_X)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
# --- 继续 ---
        epoch_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        avg_loss = epoch_loss / len(loader)
        print(f"  Epoch {epoch+1:>3}: Loss={avg_loss:.6f}")

### 4. 降维结果

运行 4. 降维结果 的代码，观察算法在该环节的行为。

In [ ]:
model.eval()
with torch.no_grad():
    X_train_encoded = model.encode(torch.FloatTensor(X_train)).numpy()
    X_test_encoded = model.encode(torch.FloatTensor(X_test)).numpy()

print(f"\n=== 降维结果 ===")
print(f"原始维度: {X_train.shape[1]}, 降维后: {X_train_encoded.shape[1]}")
print(f"训练集降维后形状: {X_train_encoded.shape}")
print(f"测试集降维后形状: {X_test_encoded.shape}")

### 5. 对比 PCA

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 自编码器 vs PCA ===")
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
# 自编码器降维
acc_ae = cross_val_score(knn, X_train_encoded, y_train, cv=5).mean()
# PCA 降维
acc_pca = cross_val_score(knn, X_train_pca, y_train, cv=5).mean()
print(f"自编码器 → 1-NN CV准确率: {acc_ae:.4f}")
print(f"PCA      → 1-NN CV准确率: {acc_pca:.4f}")
print(f"PCA 解释方差: {pca.explained_variance_ratio_.round(4)}")

### 6. 不同潜在维度

运行 6. 不同潜在维度 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同 latent_dim 对比 ===")
for ld in [1, 2, 3]:
    model_ld = Autoencoder(input_dim, 8, ld)
    opt = torch.optim.Adam(model_ld.parameters(), lr=0.01)
    for _ in range(200):
# --- 循环处理 ---
        for batch_X, _ in loader:
            recon, _ = model_ld(batch_X)
            loss = criterion(recon, batch_X)
            opt.zero_grad()
            loss.backward()
# --- 继续 ---
            opt.step()
    model_ld.eval()
    with torch.no_grad():
        X_enc = model_ld.encode(torch.FloatTensor(X_train)).numpy()
    knn_ld = KNeighborsClassifier(n_neighbors=5)
# --- 继续 ---
    acc = cross_val_score(knn_ld, X_enc, y_train, cv=5).mean()
    print(f"  latent_dim={ld}: 1-NN CV准确率={acc:.4f}")

### 7. 重建质量

运行 7. 重建质量 的代码，观察算法在该环节的行为。

In [ ]:
with torch.no_grad():
    X_recon, _ = model(torch.FloatTensor(X_test))
    recon_error = criterion(X_recon, torch.FloatTensor(X_test)).item()
print(f"\n=== 重建质量 ===")
print(f"测试集重建 MSE: {recon_error:.6f}")

print("\n=== 自编码器降维要点 ===")
print("- 编码器将高维数据压缩到低维，解码器从低维重建原始数据")
print("- 是 PCA 的非线性推广：线性自编码器 ≈ PCA")
print("- 优势：可捕捉非线性流形结构")
print("- 劣势：需要调参（网络结构、学习率、训练轮数）、不如 PCA 稳定")
# --- 输出结果 ---
print("- 可扩展：变分自编码器（VAE）可生成新样本")

## 关键代码解释

### 自编码器架构

```python

class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))

```

瓶颈层维度 latent_dim 就是降维后的维度。隐藏层 hidden_dim 提供非线性变换的能力。

### 训练过程

```python

recon, _ = model(batch_X)
loss = criterion(recon, batch_X)  # 重建误差 = MSE

```

损失函数是原始输入和重建输出之间的 MSE。解码器重建得越好，说明瓶颈层保留的信息越多。

## 使用示例

```python

# 训练后使用编码器降维
model.eval()
with torch.no_grad():
    X_encoded = model.encode(torch.FloatTensor(X)).numpy()

```

## 注意事项

1. **需要调参**：网络结构、学习率、训练轮数都需要调
2. **不如 PCA 稳定**：可能陷入局部最优
3. **小数据集上 PCA 可能更好**：自编码器的优势在大数据和非线性关系上
4. **重建损失只用 MSE**：可以用感知损失、对抗损失等

## 总结与延伸

以上代码展示了 **自编码器 降维** 的完整流程。

**解读要点**：
- 观察降维后数据的 **可分离性**——同类样本是否聚集
- 对比不同降维方法的可视化效果
- 关注 **方差解释比**（PCA）或 **邻域保持度**（UMAP）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **变分自编码器（VAE）**：学习数据的分布而非确定性映射，可以生成新样本
- **去噪自编码器（DAE）**：输入加噪声，输出是干净数据，学到的表示更鲁棒
- **稀疏自编码器**：瓶颈层加稀疏约束，学到更有意义的特征
- **卷积自编码器**：用卷积层替代全连接层，适合图像数据
